# ALS baseline for popularity calibration

This notebook documents the Spark/Scala environment for the MovieLens 1M recommendation baseline. In the study, the ALS baseline provides the unadjusted recommendations against which fixed and history-aware popularity calibration are compared.

**Current implementation:** library loading, local Spark session initialization, runtime checks, MovieLens rating loading and parsing, DataFrame inspection, and session shutdown. ALS training and recommendation generation are not implemented, and no model or recommendation results are reported.

Run the implemented code cells in order using a Scala 2.12.18 kernel with Java 17. The notebook loads Spark 3.5.9. Saved outputs record a previous execution; rerunning the cells verifies the current environment.


## 1. Verify the local environment

The environment check loads Spark, starts a local session, and counts three synthetic rows. It verifies that the notebook can execute a Spark operation independently of MovieLens data or model training.

The `$ivy` imports load the specified Spark dependencies into the Scala kernel. `spark-sql` provides structured data processing, and `spark-mllib` provides machine-learning algorithms, including ALS. Importing `SparkSession` makes the session entry point available by name.


In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession

### Initialize Spark and verify the runtime

`SparkSession.builder()` configures the application name, local execution with two worker threads, and the driver's loopback addresses. `.getOrCreate()` returns an existing valid session or creates one; `val spark` stores the returned session reference.

After initialization, the log level is set to `ERROR`. The Scala and Java versions are read from the running kernel. `spark.range(3)` creates three synthetic rows, and `.count()` executes the operation and returns their count.

The assertions require Spark 3.5.9, Scala 2.12.18, Java 17, and a row count of three. A failed assertion stops execution of the cell. The printed version numbers and count provide evidence that the environment check completed successfully. These outputs are not recommender evaluation results.


In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - ALS Baseline Environment")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")


Spark: 3.5.9


Scala: 2.12.18


Java: 17.0.16


Environment check passed: 3 synthetic rows.


spark: SparkSession = org.apache.spark.sql.SparkSession@79a2f932
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.16"
checkCount: Long = 3L

## 2. Load MovieLens ratings

The local `data/ml-1m/ratings.dat` file contains one `UserID::MovieID::Rating::Timestamp` record per line. Loading preserves all four fields. The user ID, movie ID, and observed rating are the inputs for explicit-feedback ALS; the timestamp is retained as source information.

### Define the record and parser

`Rating` defines the fields and their numeric types. `RatingParser` is a serializable function from a text line to a `Rating`. Its `apply` method splits a line at `::`, checks for four fields, and converts their values. Keep these definitions in their own cell, separate from the Spark context and data-loading variables, so the parser can be serialized without capturing driver-only notebook state.


In [3]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")

    Rating(
      fields(0).toInt,
      fields(1).toInt,
      fields(2).toFloat,
      fields(3).toLong
    )
  }
}


defined class Rating
defined class RatingParser

### Read and parse the local file

The path lookup supports a kernel working directory at either the repository root or its `notebooks` folder. It stops with a descriptive error if the dataset is missing. `sc.textFile` creates an `RDD[String]`, and `map(new RatingParser)` defines the conversion to an `RDD[Rating]`. Spark performs the parsing when an action requests the data. The source file is read without modification.


In [4]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}

val sc = spark.sparkContext
val lines = sc.textFile(ratingsPath)
val ratingsRDD = lines.map(new RatingParser)


ratingsPath: String = "../data/ml-1m/ratings.dat"
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@4b75419b
lines: org.apache.spark.rdd.RDD[String] = ../data/ml-1m/ratings.dat MapPartitionsRDD[8] at textFile at cmd4.sc:9
ratingsRDD: org.apache.spark.rdd.RDD[Rating] = MapPartitionsRDD[9] at map at cmd4.sc:10

### Convert records to a DataFrame

`import spark.implicits._` supplies the conversion helpers for `toDF()`. Spark derives the DataFrame column names and types from `Rating`. The records remain distributed; this conversion does not collect the full dataset into driver memory.


In [5]:
import spark.implicits._
val ratingsDF = ratingsRDD.toDF()


import spark.implicits._
ratingsDF: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]

### Inspect the schema and sample rows

`printSchema()` displays the column structure. `show(5, truncate = false)` executes enough processing to display five records without shortening their contents. This preview checks the loading and conversion of sample rows; it is not a full-dataset validation or a model evaluation. No training split is created here.


In [6]:
ratingsDF.printSchema()
ratingsDF.show(5, truncate = false)


root
 |-- userId: integer (nullable = false)
 |-- movieId: integer (nullable = false)
 |-- rating: float (nullable = false)
 |-- timestamp: long (nullable = false)



+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1193   |5.0   |978300760|
|1     |661    |3.0   |978302109|
|1     |914    |3.0   |978301968|
|1     |3408   |4.0   |978300275|
|1     |2355   |5.0   |978824291|
+------+-------+------+---------+
only showing top 5 rows



## 3. ALS baseline

**Status:** Not implemented. This section contains no model-training code or fitted-model results.


## 4. Baseline recommendations

**Status:** Not implemented. No candidate rankings or recommendation lists have been generated.


## 5. Stop the Spark session

`spark.stop()` shuts down the Spark session and its underlying Spark context, releasing their resources. Run this cell after data inspection and any subsequent Spark work. Further Spark operations require initializing a session again.


In [7]:
spark.stop()
println("Spark session stopped.")


Spark session stopped.
